In [ ]:
annual = pd.read_csv(DATA/'generation_annual_summary_2010_2025.csv')
annual_long = pd.read_csv(DATA/'generation_annual_long_2010_2025.csv')
monthly = pd.read_csv(DATA/'generation_monthly_2010_2025.csv')
cso_2025 = pd.read_csv(DATA/'cso_supply_monthly_2025.csv')
thermal = pd.read_csv(DATA/'thermal_generation_efficiency_2010_2025.csv')
fuel_supply = pd.read_csv(DATA/'fuel_supply_dependency_2010_2025.csv')
validation = pd.read_csv(DATA/'generation_source_validation_2023_2025.csv')
concentration = pd.read_csv(DATA/'generation_concentration_2010_2025.csv')
relationships = pd.read_csv(DATA/'monthly_operating_relationships_by_year.csv')
seasonality = pd.read_csv(DATA/'seasonality_profile_2021_2025.csv')
ops_2025 = pd.read_csv(DATA/'monthly_grid_operating_metrics_2025.csv')
ress = pd.read_csv(DATA/'ress_auction_price_history_2020_2025.csv')
gas_cost = pd.read_csv(DATA/'gas_fuel_carbon_cost_sensitivity_2025.csv')
evidence = pd.read_csv(DATA/'project04_evidence_hierarchy.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/generation_annual_summary_2010_2025.csv'

In [ ]:
annual = pd.read_csv(DATA/'generation_annual_summary_2010_2025.csv')
annual_long = pd.read_csv(DATA/'generation_annual_long_2010_2025.csv')
monthly = pd.read_csv(DATA/'generation_monthly_2010_2025.csv')
cso_2025 = pd.read_csv(DATA/'cso_supply_monthly_2025.csv')
thermal = pd.read_csv(DATA/'thermal_generation_efficiency_2010_2025.csv')
fuel_supply = pd.read_csv(DATA/'fuel_supply_dependency_2010_2025.csv')
validation = pd.read_csv(DATA/'generation_source_validation_2023_2025.csv')
concentration = pd.read_csv(DATA/'generation_concentration_2010_2025.csv')
relationships = pd.read_csv(DATA/'monthly_operating_relationships_by_year.csv')
seasonality = pd.read_csv(DATA/'seasonality_profile_2021_2025.csv')
ops_2025 = pd.read_csv(DATA/'monthly_grid_operating_metrics_2025.csv')
ress = pd.read_csv(DATA/'ress_auction_price_history_2020_2025.csv')
gas_cost = pd.read_csv(DATA/'gas_fuel_carbon_cost_sensitivity_2025.csv')
evidence = pd.read_csv(DATA/'project04_evidence_hierarchy.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/generation_annual_summary_2010_2025.csv'

# Ireland Electricity Generation Analytics
## Project 04 — How Ireland changed what powers its electricity system

**Research question**

> How did Ireland transform its electricity-generation system between 2010 and 2025, and what does that transition mean for generation mix, operating flexibility, thermal efficiency, imported-fuel exposure, carbon cost and generation economics?

This notebook separates observed data, derived metrics and interpretation. It does not treat correlation as causation or compare unlike cost concepts as though they were identical.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Updated to point to the local workspace directory
DATA = Path('data/processed')
CHARTS = Path('charts')
CHARTS.mkdir(exist_ok=True)

In [20]:
import urllib.request
import zipfile
import shutil
from pathlib import Path

# Ensure clean target directory exists
DATA.mkdir(parents=True, exist_ok=True)

temp_zip = Path('repo.zip')
temp_dir = Path('repo_extracted')

try:
    if temp_zip.exists():
        temp_zip.unlink()
    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    # Add a user-agent to bypass potential raw github/api request blocks
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
    urllib.request.install_opener(opener)

    # Try to download the repository archive (trying both main and master branches)
    downloaded = False
    for branch in ['main', 'master']:
        try:
            print(f"Downloading repository archive from {branch} branch...")
            zip_url = f"https://github.com/b-g/ireland-electricity-transition/archive/refs/heads/{branch}.zip"
            urllib.request.urlretrieve(zip_url, temp_zip)
            downloaded = True
            break
        except Exception as e:
            print(f"Failed to download ZIP for {branch} branch: {e}")

    if not downloaded:
        raise RuntimeError("Failed to download repository ZIP from both main and master branches.")

    # Extract ZIP archive
    print("Extracting archive...")
    with zipfile.ZipFile(temp_zip, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)

    # Locate the nested processed data folder in the extracted directory
    print("Locating and moving CSV files...")
    csv_files = list(temp_dir.glob('**/projects/04_generation/data/processed/*.csv'))

    # Fallback to look for any processed/ CSVs if the nested folder structure differs
    if not csv_files:
        csv_files = list(temp_dir.glob('**/data/processed/*.csv'))

    count = 0
    for csv_file in csv_files:
        shutil.copy(csv_file, DATA / csv_file.name)
        count += 1

    if count > 0:
        print(f"Successfully retrieved and placed {count} CSV files in {DATA}!")
    else:
        print("No CSV files found in the expected folders of the extracted archive.")

except Exception as e:
    print(f"An error occurred during repository extraction: {e}")
finally:
    # Clean up temporary resources
    if temp_zip.exists():
        temp_zip.unlink()
    if temp_dir.exists():
        shutil.rmtree(temp_dir)

Failed to download ZIP for main branch: HTTP Error 404: Not Found
Failed to download ZIP for master branch: HTTP Error 404: Not Found
An error occurred during repository extraction: Failed to download repository ZIP from both main and master branches.


## 1. Load canonical datasets


In [21]:
annual = pd.read_csv(DATA/'generation_annual_summary_2010_2025.csv')
annual_long = pd.read_csv(DATA/'generation_annual_long_2010_2025.csv')
monthly = pd.read_csv(DATA/'generation_monthly_2010_2025.csv')
cso_2025 = pd.read_csv(DATA/'cso_supply_monthly_2025.csv')
thermal = pd.read_csv(DATA/'thermal_generation_efficiency_2010_2025.csv')
fuel_supply = pd.read_csv(DATA/'fuel_supply_dependency_2010_2025.csv')
validation = pd.read_csv(DATA/'generation_source_validation_2023_2025.csv')
concentration = pd.read_csv(DATA/'generation_concentration_2010_2025.csv')
relationships = pd.read_csv(DATA/'monthly_operating_relationships_by_year.csv')
seasonality = pd.read_csv(DATA/'seasonality_profile_2021_2025.csv')
ops_2025 = pd.read_csv(DATA/'monthly_grid_operating_metrics_2025.csv')
ress = pd.read_csv(DATA/'ress_auction_price_history_2020_2025.csv')
gas_cost = pd.read_csv(DATA/'gas_fuel_carbon_cost_sensitivity_2025.csv')
evidence = pd.read_csv(DATA/'project04_evidence_hierarchy.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/generation_annual_summary_2010_2025.csv'

## 2. Source reconciliation
SEAI monthly generation aggregates are checked against CSO annual grid-scale generation for 2023–2025. The series reconcile extremely closely, supporting use of the SEAI export as the 2010–2025 backbone while retaining CSO as the recent operational/validation layer.


In [ ]:
validation.groupby('year').agg(
    max_abs_pct_difference=('difference_pct_of_cso', lambda s: s.abs().max()),
    mean_abs_pct_difference=('difference_pct_of_cso', lambda s: s.abs().mean())
)


## 3. Structural transition — generation changed composition more than scale


In [ ]:
a = annual.set_index('year')
summary = pd.DataFrame({
    '2010': [a.loc[2010,'total_generation_gwh'], a.loc[2010,'gas_share_pct'], a.loc[2010,'wind_share_pct'], a.loc[2010,'coal_share_pct']],
    '2025': [a.loc[2025,'total_generation_gwh'], a.loc[2025,'gas_share_pct'], a.loc[2025,'wind_share_pct'], a.loc[2025,'coal_share_pct']],
}, index=['Total generation GWh','Gas share %','Wind share %','Coal share %'])
summary


In [ ]:
mix = annual_long.pivot(index='year',columns='source',values='generation_gwh').fillna(0)
mix[['Gas','Wind','Coal','Peat & Biomass','Solar','Hydro']].plot(figsize=(12,7), marker='o')
plt.title('Ireland domestic electricity generation by major source, 2010–2025')
plt.ylabel('GWh')
plt.show()


## 4. Concentration — the system increasingly revolves around its two largest sources


In [ ]:
concentration[['year','top_1_source','top_1_share_pct','top_2_source','top_2_share_pct','top_2_combined_share_pct','effective_number_of_equal_sources']].tail(8)


## 5. Operating relationship — wind and gas are not a simple one-for-one seesaw
Monthly gas–wind correlation varies substantially by year. Negative correlation in some years is consistent with gas reducing output when wind is high, but correlation alone does not identify balancing causally; demand, imports, outages, storage, prices and other sources also matter.


In [ ]:
relationships[['year','gas_wind_corr','wind_solar_corr','gas_monthly_cv','wind_monthly_cv']].tail(10)


## 6. Seasonality — wind and solar have different profiles


In [ ]:
seasonality[['month','Gas_mean','Wind_mean','Solar_mean']]


## 7. Recent grid supply — imports and storage


In [ ]:
ops_2025[['period','generation_gwh','net_imports_gwh','storage_discharge_gwh','net_import_share_proxy_pct']].round(2)


## 8. Thermal generation — less fuel, better efficiency, more gas concentration


In [ ]:
thermal[['year','public_thermal_input_ktoe','public_thermal_electricity_output_ktoe','public_thermal_efficiency_pct','gas_input_share_pct','coal_input_share_pct','peat_input_share_pct']].tail(10)


## 9. Fuel origin — domestic generation is not the same as indigenous energy


In [ ]:
fuel_supply.query('year == 2025')[['fuel','indigenous_production_ktoe','imports_ktoe','primary_energy_requirement_ktoe','import_share_of_indigenous_plus_imports_pct']].sort_values('imports_ktoe',ascending=False)


## 10. Renewable investment benchmarks — RESS history
RESS prices are successful-auction revenue/support benchmarks. They are not dispatch costs and should not be interpreted as plant marginal cost or a universal LCOE.


In [ ]:
ress[['auction','auction_year','all_projects_eur_mwh','wind_eur_mwh','solar_eur_mwh']]


## 11. Gas economics — sensitivity, not false precision
The gas model uses:
- SEAI 2025 provisional natural-gas combustion factor: 203.5 gCO₂/kWh fuel (NCV).
- 2025 EU ETS average auction price: €73.43/tCO₂.
- A range of gas fuel prices and thermal efficiencies.

It excludes plant-specific O&M, startup costs, capacity payments, financing, hedging and network charges.


In [ ]:
gas_cost.query('efficiency_pct == 50')


## 12. Evidence hierarchy


In [ ]:
evidence


## Reproducibility and source notes

The repository keeps original-source data separate from analytical outputs. The public release includes the processed datasets required to reproduce the figures in this notebook.

Primary evidence:
- Central Statistics Office (CSO), *Electricity Supply by Source*.
- Sustainable Energy Authority of Ireland (SEAI), electricity-generation statistics, National Energy Balance and Energy by Fuel.
- EirGrid, RESS final auction results.
- European Commission, EU ETS auction reporting.
- Gas Networks Ireland, imbalance-price / SAP documentation.

Important cost interpretation:
- RESS strike prices are auction/support revenue benchmarks.
- The gas model is a fuel + carbon sensitivity, not a claim about any plant's private procurement or total LCOE.
- Gas Networks Ireland's SAP/imbalance framework and the official ONS/National Gas Transmission SAP series are used as market context, not as a substitute for plant-specific contracts.


## 13. Conclusion

Ireland's electricity transition between 2010 and 2025 was primarily a transformation in **composition**, not a dramatic expansion of domestic generation.

Wind expanded from a secondary source into one of the two dominant sources. Coal and peat retreated sharply, solar began scaling, and public thermal generation became smaller and more efficient.

Yet the transition did not eliminate external exposure. Gas remains the dominant thermal fuel and is mostly imported. Direct electricity imports also rose sharply between 2023 and 2025. The system is therefore increasingly built around a combination of **indigenous variable renewables, gas flexibility, interconnection and emerging storage**.

The most defensible economic comparison is not a single 'cost per MWh' league table. Gas economics are highly sensitive to fuel and carbon prices, while RESS prices are long-term support/revenue benchmarks for renewable investment. The distinction is essential.

This naturally leads to Project 05: **what electricity does Ireland import, where does it come from, and what does it cost?**
